# 🏦 CréditoBrasil — Consultas em Linguagem Natural

**Arquitetura:**  Pergunta em português → DSPy gera SQL → SQLite executa → Resposta em português

**Modelo:** `qwen2.5:3b` via Ollama (3 B parâmetros, ótimo para SQL)  
**Alternativas:** `llama3.2:3b`, `gemma3:4b`, `phi4-mini:3.8b`

**Banco de dados:** `creditoBrasil.db`  
- 191 indicadores macroeconômicos e de crédito  
- 35 mil registros temporais (2016 → 2026)  
- Cobertura nacional e por estado (26 UFs + DF)

---

## Como usar
1. Execute todas as células em ordem (**Runtime → Run all**)  
2. Faça o upload do arquivo `creditoBrasil.db` quando solicitado  
3. Use o chat interativo na última célula

---
## ⚙️ Célula 1 — Instalação do Ollama e dependências

In [ ]:
# Instala Ollama
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Instala pacotes Python
!pip install -q dspy ollama requests ipywidgets

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ol

---
## 🚀 Célula 2 — Iniciar servidor Ollama + baixar modelo

In [ ]:
import subprocess
import threading
import time
import requests

# ─── Configuração do modelo ──────────────────────────────────────────────────
# Escolha um modelo abaixo (descomente apenas UM):
MODELO = "qwen2.5:3b"       # ✅ Recomendado — melhor para SQL entre modelos ≤4B
# MODELO = "llama3.2:3b"    # Alternativa — bom para português
# MODELO = "gemma3:4b"      # Alternativa — 4B parâmetros, Google
# MODELO = "phi4-mini"      # Alternativa — 3.8B, Microsoft
# ─────────────────────────────────────────────────────────────────────────────

def _run_ollama_serve():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)

# Sobe o servidor em background
threading.Thread(target=_run_ollama_serve, daemon=True).start()

print("⏳ Aguardando Ollama...", end="")
for _ in range(20):
    try:
        r = requests.get("http://localhost:11434/", timeout=2)
        if r.status_code == 200:
            print(" ✅ Online!")
            break
    except Exception:
        pass
    time.sleep(2)
    print(".", end="", flush=True)
else:
    print(" ❌ Timeout — tente executar novamente.")

print(f"\n📥 Baixando modelo '{MODELO}'...")
!ollama pull {MODELO}
print(f"✅ Modelo '{MODELO}' pronto!")

⏳ Aguardando Ollama.... ✅ Online!

📥 Baixando modelo 'qwen2.5:3b'...

✅ Modelo 'qwen2.5:3b' pronto!


---
## 📂 Célula 3 — Upload do banco de dados

In [ ]:
import os
from google.colab import files

DB_PATH = "creditoBrasil.db"

if not os.path.exists(DB_PATH):
    print("📎 Faça o upload do arquivo 'creditoBrasil.db':")
    uploaded = files.upload()
    if DB_PATH not in uploaded:
        raise FileNotFoundError(
            f"Arquivo '{DB_PATH}' não encontrado. "
            "Certifique-se de fazer upload com esse nome exato."
        )
    print(f"✅ '{DB_PATH}' carregado com sucesso!")
else:
    print(f"✅ '{DB_PATH}' já está disponível.")

✅ 'creditoBrasil.db' já está disponível.


---
## 🗄️ Célula 4 — Conexão com o banco e utilitários

In [ ]:
import sqlite3
import json
from textwrap import dedent
from typing import Optional

# ─── Conexão global ──────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row          # retorna dicts-like rows
print(f"✅ Conectado a '{DB_PATH}'")


def executar_sql(sql: str, max_rows: int = 50):
    """Executa um SQL e retorna (colunas, linhas, erro)."""
    try:
        # Limpa markdown code-fences que o modelo às vezes insere
        sql_limpo = sql.strip()
        for tag in ["```sql", "```sqlite", "```"]:
            sql_limpo = sql_limpo.replace(tag, "")
        sql_limpo = sql_limpo.strip().rstrip(";")

        cur = conn.execute(sql_limpo)
        rows = cur.fetchmany(max_rows)
        colunas = [d[0] for d in cur.description] if cur.description else []
        return colunas, [dict(zip(colunas, r)) for r in rows], None
    except Exception as e:
        return [], [], str(e)


def resumo_bd():
    """Retorna um snapshot rápido do banco para diagnóstico."""
    _, rows, _ = executar_sql(
        "SELECT name, (SELECT COUNT(*) FROM sqlite_master m2 "
        "WHERE m2.type='table' LIMIT 1) FROM sqlite_master "
        "WHERE type='table' ORDER BY name"
    )
    tabelas = [r["name"] for r in rows] if rows else []
    print("Tabelas encontradas:", tabelas)

    for t in ["dim_serie", "dim_uf", "fact_serie_temporal"]:
        _, r2, _ = executar_sql(f"SELECT COUNT(*) as n FROM {t}")
        print(f"  {t}: {r2[0]['n']} linhas")

resumo_bd()

✅ Conectado a 'creditoBrasil.db'
Tabelas encontradas: ['consulta_ia', 'dim_serie', 'dim_uf', 'fact_serie_temporal', 'fact_simulacao_risco', 'log_auditoria', 'ranking_oportunidade', 'sessao', 'sqlite_sequence', 'usuario']
  dim_serie: 191 linhas
  dim_uf: 27 linhas
  fact_serie_temporal: 35030 linhas


---
## 🧠 Célula 5 — Configuração do DSPy + Assinaturas

In [ ]:
import dspy

# ─── LM via Ollama ───────────────────────────────────────────────────────────
lm = dspy.LM(
    model=f"ollama/{MODELO}",
    api_base="http://localhost:11434",
    max_tokens=600,
    temperature=0.0,          # determinístico para SQL
    cache=False,
)
dspy.configure(lm=lm)
print(f"✅ DSPy configurado com '{MODELO}'")


# ─── Schema detalhado do banco ───────────────────────────────────────────────
SCHEMA = dedent("""
    Banco SQLite: creditoBrasil.db  (dados econômicos e de crédito do Brasil, 2016-2026)

    TABELAS PRINCIPAIS:

    dim_uf  — estados brasileiros
      sigla_uf  TEXT  PK  (ex: 'SP', 'RJ', 'MG', 'BA'...)
      nome      TEXT      (ex: 'São Paulo', 'Rio de Janeiro')
      codigo_ibge TEXT
      regiao_br TEXT      ('Norte','Nordeste','Centro-Oeste','Sudeste','Sul')

    dim_serie  — catálogo de 191 indicadores
      id_serie       INTEGER  PK
      nome_indicador TEXT     (ex: 'Inadimplência PF - SP', 'Saldo Total - RJ', 'Selic - Meta')
      categoria      TEXT     ('credito','taxas_de_juros','inflacao_precos','cambio',
                               'atividade_economica','mercado_financeiro','emprego_e_renda')
      subcategoria   TEXT     ('inadimplencia','saldos_credito','taxas_juros','endividamento',
                               'concessoes','selic','ipca_geral','igp','dolar','bolsa','pib',
                               'mercado_trabalho_pnad','poupanca','referencias')
      periodicidade  TEXT     ('diária','mensal','trimestral','anual')
      unidade_medida TEXT     ('%','R$ mi','% a.a.','% a.m.','p.p.','R$','pontos')
      abrangencia    TEXT     ('Brasil' ou sigla do estado, ex: 'SP')
      descricao      TEXT
      ativo          INTEGER

    fact_serie_temporal  — 35.030 observações históricas
      id_serie        INTEGER  FK→dim_serie.id_serie
      sigla_uf        TEXT     (sigla do estado, ou NULL para indicadores nacionais)
      data_referencia TEXT     (formato 'YYYY-MM-DD')
      valor           REAL
      data_ingestao   TEXT

    INDICADORES DISPONÍVEIS POR ESTADO (cada UF tem séries próprias):
      • Inadimplência PF - <UF>      categoria='credito', subcategoria='inadimplencia', unidade='%'
      • Inadimplência PJ - <UF>      categoria='credito', subcategoria='inadimplencia', unidade='%'
      • Inadimplência Total - <UF>   categoria='credito', subcategoria='inadimplencia', unidade='%'
      • Saldo PF - <UF>              categoria='credito', subcategoria='saldos_credito', unidade='R$ mi'
      • Saldo PJ - <UF>              categoria='credito', subcategoria='saldos_credito', unidade='R$ mi'
      • Saldo Total - <UF>           categoria='credito', subcategoria='saldos_credito', unidade='R$ mi'

    INDICADORES NACIONAIS (sigla_uf IS NULL em fact_serie_temporal):
      • Inadimplência PF / PJ / total  (%)
      • Saldo crédito PF / PJ          (R$ mi)
      • Concessões crédito PF/PJ/total (R$ mi)
      • Endividamento das famílias / Comprometimento de renda PF  (%)
      • Taxa de juros média PF/PJ      (% a.a.)
      • Spread bancário total          (p.p.)
      • Selic - Meta / Taxa diária / Acumulada  (% a.a. / % a.m.)
      • IPCA - Variação mensal / Acumulado 12m  (%)
      • IGP-M                          (% a.m.)
      • Dólar (venda) diário / PTAX    (R$)
      • Ibovespa                       (pontos)
      • PIB anual / trimestral         (R$ mi)
      • Taxa de desemprego (PNAD)      (%)
      • Poupança - Rentabilidade       (% a.m.)
      • TR - Taxa Referencial          (% a.m.)

    REGRAS IMPORTANTES:
      1. Sempre use JOIN entre fact_serie_temporal e dim_serie via id_serie.
      2. Para indicadores estaduais, filtre por sigla_uf na fact_serie_temporal
         OU use LIKE no nome_indicador (ex: nome_indicador LIKE 'Inadimplência PF - SP').
      3. Indicadores nacionais têm sigla_uf IS NULL.
      4. Siglas de estado: AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO
      5. 'São Paulo' → sigla_uf = 'SP'; 'Rio de Janeiro' → 'RJ'; 'Minas Gerais' → 'MG'.
      6. Para dados recentes use ORDER BY data_referencia DESC LIMIT N.
      7. Para comparar estados, use GROUP BY sigla_uf.
      8. Nunca use * em SELECT — liste as colunas necessárias.
      9. Retorne apenas o SQL, sem explicações.
""").strip()


# ─── Assinaturas DSPy ────────────────────────────────────────────────────────
class GerarSQL(dspy.Signature):
    """Você é um especialista em SQLite. Com base no schema fornecido, gere
    exatamente um comando SELECT válido para responder à pergunta do usuário.
    Retorne APENAS o SQL, sem markdown, sem explicações."""

    schema: str     = dspy.InputField(desc="Schema completo do banco de dados")
    question: str   = dspy.InputField(desc="Pergunta em linguagem natural (pode ser em português ou inglês)")
    sql_query: str  = dspy.OutputField(desc="Comando SELECT SQLite válido, sem markdown")


class RefinirSQL(dspy.Signature):
    """Corrija o SQL com base no erro retornado pelo banco.
    Retorne APENAS o SQL corrigido, sem markdown."""

    schema: str     = dspy.InputField(desc="Schema do banco")
    question: str   = dspy.InputField(desc="Pergunta original")
    sql_query: str  = dspy.InputField(desc="SQL com problema")
    error: str      = dspy.InputField(desc="Mensagem de erro SQLite")
    sql_query: str  = dspy.OutputField(desc="SQL corrigido")


class GerarResposta(dspy.Signature):
    """Formule uma resposta clara e completa em português brasileiro para o usuário.
    Interprete os dados retornados pelo banco, destaque os valores mais relevantes
    e, quando possível, forneça contexto ou insight sobre os números."""

    question: str       = dspy.InputField(desc="Pergunta original do usuário")
    sql_query: str      = dspy.InputField(desc="SQL que foi executado")
    data: str           = dspy.InputField(desc="Dados retornados pelo banco (JSON)")
    answer: str         = dspy.OutputField(desc="Resposta em português, clara e informativa")


print("✅ Assinaturas DSPy definidas")

✅ DSPy configurado com 'qwen2.5:3b'
✅ Assinaturas DSPy definidas


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "GerarSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)
/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "RefinirSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


---
## 📚 Célula 6 — Exemplos de treinamento (few-shot)

In [ ]:
EXEMPLOS_SQL = [
    # ── Inadimplência por UF ─────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual a inadimplência de pessoas físicas em São Paulo no último mês disponível?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE fst.sigla_uf = 'SP' "
            "AND ds.nome_indicador = 'Inadimplência PF - SP' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Compare a inadimplência total de todos os estados no mês mais recente.",
        sql_query=(
            "SELECT fst.sigla_uf, du.nome, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "JOIN dim_uf du ON du.sigla_uf = fst.sigla_uf "
            "WHERE ds.subcategoria = 'inadimplencia' "
            "AND ds.nome_indicador LIKE 'Inadimplência Total - %' "
            "AND fst.data_referencia = ("
            "  SELECT MAX(data_referencia) FROM fact_serie_temporal fst2 "
            "  JOIN dim_serie ds2 ON ds2.id_serie = fst2.id_serie "
            "  WHERE ds2.nome_indicador LIKE 'Inadimplência Total - %'"
            ") "
            "ORDER BY fst.valor DESC"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quais os 5 estados com maior inadimplência PJ atualmente?",
        sql_query=(
            "SELECT fst.sigla_uf, du.nome, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "JOIN dim_uf du ON du.sigla_uf = fst.sigla_uf "
            "WHERE ds.nome_indicador LIKE 'Inadimplência PJ - %' "
            "AND fst.data_referencia = ("
            "  SELECT MAX(data_referencia) FROM fact_serie_temporal fst2 "
            "  JOIN dim_serie ds2 ON ds2.id_serie = fst2.id_serie "
            "  WHERE ds2.nome_indicador LIKE 'Inadimplência PJ - %'"
            ") "
            "ORDER BY fst.valor DESC LIMIT 5"
        )
    ).with_inputs("schema", "question"),

    # ── Saldo de crédito ─────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual o saldo total de crédito em Minas Gerais?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE fst.sigla_uf = 'MG' "
            "AND ds.nome_indicador = 'Saldo Total - MG' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Mostre a evolução do saldo de crédito PF no Nordeste nos últimos 12 meses.",
        sql_query=(
            "SELECT fst.sigla_uf, du.nome, fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "JOIN dim_uf du ON du.sigla_uf = fst.sigla_uf "
            "WHERE ds.nome_indicador LIKE 'Saldo PF - %' "
            "AND du.regiao_br = 'Nordeste' "
            "AND fst.data_referencia >= date('now','-12 months') "
            "ORDER BY fst.sigla_uf, fst.data_referencia"
        )
    ).with_inputs("schema", "question"),

    # ── Selic / juros ────────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual a taxa Selic atual (meta)?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Selic - Meta' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Como a Selic evoluiu nos últimos 24 meses?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Selic - Meta' "
            "AND fst.data_referencia >= date('now','-24 months') "
            "ORDER BY fst.data_referencia"
        )
    ).with_inputs("schema", "question"),

    # ── IPCA / inflação ──────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual a inflação (IPCA) acumulada nos últimos 12 meses?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'IPCA - Acumulado 12 meses' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Endividamento ────────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual o endividamento atual das famílias brasileiras?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Endividamento das famílias' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Concessões ───────────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Quais as concessões de crédito para pessoas físicas no último mês?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Concessões crédito PF' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Comparativo regional ─────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual região do Brasil tem a maior inadimplência média de PF?",
        sql_query=(
            "SELECT du.regiao_br, AVG(fst.valor) AS inadimplencia_media "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "JOIN dim_uf du ON du.sigla_uf = fst.sigla_uf "
            "WHERE ds.nome_indicador LIKE 'Inadimplência PF - %' "
            "AND fst.data_referencia = ("
            "  SELECT MAX(data_referencia) FROM fact_serie_temporal fst2 "
            "  JOIN dim_serie ds2 ON ds2.id_serie = fst2.id_serie "
            "  WHERE ds2.nome_indicador LIKE 'Inadimplência PF - %'"
            ") "
            "GROUP BY du.regiao_br "
            "ORDER BY inadimplencia_media DESC"
        )
    ).with_inputs("schema", "question"),

    # ── Câmbio ───────────────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual o valor do dólar hoje?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Dólar PTAX (venda)' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Desemprego / PIB ─────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual a taxa de desemprego atual no Brasil?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Taxa de desemprego (PNAD)' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Spread ───────────────────────────────────────────────────────────────
    dspy.Example(
        schema=SCHEMA,
        question="Qual o spread bancário total atual?",
        sql_query=(
            "SELECT fst.data_referencia, fst.valor "
            "FROM fact_serie_temporal fst "
            "JOIN dim_serie ds ON ds.id_serie = fst.id_serie "
            "WHERE ds.nome_indicador = 'Spread bancário total' "
            "ORDER BY fst.data_referencia DESC LIMIT 1"
        )
    ).with_inputs("schema", "question"),
]

print(f"✅ {len(EXEMPLOS_SQL)} exemplos de treinamento carregados")

✅ 14 exemplos de treinamento carregados


---
## 🤖 Célula 7 — Módulo principal Text-to-SQL

In [ ]:
class CreditoBrasilQA(dspy.Module):
    """
    Pipeline completo:
      1. Gera SQL a partir de pergunta em linguagem natural
      2. Executa no banco SQLite
      3. Refina o SQL se houver erro (até 2 tentativas)
      4. Converte os resultados em resposta em português
    """

    def __init__(self, exemplos: list):
        super().__init__()
        self.gerador = dspy.ChainOfThought(GerarSQL)
        self.refinador = dspy.ChainOfThought(RefinirSQL)
        self.respondedor = dspy.ChainOfThought(GerarResposta)

        # Injeta exemplos few-shot
        self.gerador.demos = exemplos

    def _limpar_sql(self, texto: str) -> str:
        """Remove markdown e espaços extras do SQL gerado."""
        for tag in ["```sql", "```sqlite", "```"]:
            texto = texto.replace(tag, "")
        # Pega apenas o primeiro SELECT se o modelo gerar múltiplos
        linhas = [l.strip() for l in texto.strip().splitlines() if l.strip()]
        sql_lines = []
        capturando = False
        for linha in linhas:
            if linha.upper().startswith("SELECT"):
                capturando = True
            if capturando:
                sql_lines.append(linha)
        return " ".join(sql_lines).rstrip(";")

    def forward(self, question: str, verbose: bool = False):
        # ── Passo 1: gerar SQL ────────────────────────────────────────────────
        out = self.gerador(schema=SCHEMA, question=question)
        sql = self._limpar_sql(out.sql_query)

        if verbose:
            print(f"\n🔍 SQL gerado:\n{sql}\n")

        # ── Passo 2: executar ─────────────────────────────────────────────────
        colunas, rows, erro = executar_sql(sql)

        # ── Passo 3: refinamento em caso de erro ──────────────────────────────
        tentativas = 0
        while erro and tentativas < 2:
            tentativas += 1
            if verbose:
                print(f"⚠️  Erro (tentativa {tentativas}): {erro}")
            refin = self.refinador(
                schema=SCHEMA,
                question=question,
                sql_query=sql,
                error=erro,
            )
            sql = self._limpar_sql(refin.sql_query)
            if verbose:
                print(f"🔧 SQL refinado (tentativa {tentativas}):\n{sql}\n")
            colunas, rows, erro = executar_sql(sql)

        if erro:
            return dspy.Prediction(
                sql_query=sql,
                data=[],
                answer=f"❌ Não foi possível executar a consulta após {tentativas} tentativas.\nErro: {erro}",
                error=erro,
            )

        # ── Passo 4: gerar resposta em português ──────────────────────────────
        dados_json = json.dumps(rows[:20], ensure_ascii=False, default=str)
        resp = self.respondedor(
            question=question,
            sql_query=sql,
            data=dados_json,
        )

        return dspy.Prediction(
            sql_query=sql,
            data=rows,
            answer=resp.answer,
            error=None,
        )


# Instancia o agente
agente = CreditoBrasilQA(exemplos=EXEMPLOS_SQL)
print("✅ Agente CreditoBrasilQA pronto!")

✅ Agente CreditoBrasilQA pronto!


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


---
## 🧪 Célula 8 — Testes automáticos

In [ ]:
PERGUNTAS_TESTE = [
    "Qual a inadimplência de pessoas físicas em São Paulo no último mês?",
    "Quais os 5 estados com maior inadimplência total atualmente?",
    "Qual região tem maior saldo de crédito PF?",
    "Qual a taxa Selic atual?",
    "Como o IPCA evoluiu nos últimos 6 meses?",
    "Qual o valor do dólar mais recente?",
    "Qual o saldo total de crédito no Rio de Janeiro?",
    "Qual a taxa de desemprego atual no Brasil?",
]

SEPARATOR = "─" * 70

for i, pergunta in enumerate(PERGUNTAS_TESTE, 1):
    print(f"\n{SEPARATOR}")
    print(f"❓ [{i}/{len(PERGUNTAS_TESTE)}] {pergunta}")
    print(SEPARATOR)

    resultado = agente(question=pergunta, verbose=False)

    print(f"📊 SQL: {resultado.sql_query}")
    print(f"📋 Linhas retornadas: {len(resultado.data)}")
    print(f"\n💬 Resposta:\n{resultado.answer}")

print(f"\n{SEPARATOR}")
print("✅ Testes concluídos!")


──────────────────────────────────────────────────────────────────────
❓ [1/8] Qual a inadimplência de pessoas físicas em São Paulo no último mês?
──────────────────────────────────────────────────────────────────────
📊 SQL: SELECT f.sigla_uf, f.data_referencia, f.valor FROM fact_serie_temporal f JOIN dim_serie d ON f.id_serie = d.id_serie WHERE d.nome_indicador LIKE 'Inadimplência PF - SP' AND f.sigla_uf = 'SP' ORDER BY f.data_referencia DESC LIMIT 1
📋 Linhas retornadas: 1

💬 Resposta:
A inadimplência de pessoas físicas em São Paulo no último mês (até 31/01/2026) foi de 4,73%. Este valor representa a taxa percentual de inadimplemento das contas de pessoas físicas na cidade de São Paulo.

──────────────────────────────────────────────────────────────────────
❓ [2/8] Quais os 5 estados com maior inadimplência total atualmente?
──────────────────────────────────────────────────────────────────────
📊 SQL: SELECT dim_uf.sigla_uf, SUM(fact_serie_temporal.valor) as inadimplencia_total FROM 

---
## 💬 Célula 9 — Chat interativo

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets

# ─── Widget de chat ───────────────────────────────────────────────────────────
HISTORICO: list[dict] = []
VERBOSE = False   # mude para True para ver o SQL gerado

# Cria os controles
input_box = widgets.Text(
    placeholder="Digite sua pergunta sobre crédito, inadimplência, juros, câmbio...",
    layout=widgets.Layout(width="85%"),
)
btn_enviar = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="12%"),
)
btn_sql = widgets.ToggleButton(
    value=False,
    description="Mostrar SQL",
    button_style="",
    layout=widgets.Layout(width="12%"),
)
btn_limpar = widgets.Button(
    description="Limpar",
    button_style="warning",
    layout=widgets.Layout(width="10%"),
)
output = widgets.Output()

def renderizar_historico():
    output.clear_output()
    with output:
        for item in HISTORICO:
            display(HTML(
                f"""<div style='background:#e8f4fd;border-left:4px solid #2196F3;
                     padding:10px;margin:6px 0;border-radius:4px'>
                    <b>❓ Você:</b> {item['pergunta']}
                </div>"""
            ))
            if item.get('sql') and btn_sql.value:
                display(HTML(
                    f"""<div style='background:#f5f5f5;border-left:4px solid #9E9E9E;
                         padding:8px;margin:4px 0;border-radius:4px;
                         font-family:monospace;font-size:12px'>
                        <b>🔍 SQL:</b> {item['sql']}
                    </div>"""
                ))
            cor = "#e8f5e9" if not item.get('erro') else "#ffebee"
            borda = "#4CAF50" if not item.get('erro') else "#F44336"
            display(HTML(
                f"""<div style='background:{cor};border-left:4px solid {borda};
                     padding:10px;margin:6px 0;border-radius:4px'>
                    <b>💬 Agente:</b><br>{item['resposta'].replace(chr(10),'<br>')}
                </div>"""
            ))

def ao_enviar(_):
    global VERBOSE
    pergunta = input_box.value.strip()
    if not pergunta:
        return
    input_box.value = ""
    VERBOSE = btn_sql.value

    with output:
        display(HTML("<i>⏳ Processando...</i>"))

    resultado = agente(question=pergunta, verbose=VERBOSE)

    HISTORICO.append({
        "pergunta": pergunta,
        "sql": resultado.sql_query,
        "resposta": resultado.answer,
        "erro": bool(resultado.error),
    })
    renderizar_historico()

def ao_limpar(_):
    HISTORICO.clear()
    output.clear_output()

btn_enviar.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
input_box.on_submit(ao_enviar)

# Layout
display(HTML("""
<div style='background:linear-gradient(135deg,#1565C0,#0288D1);
     color:white;padding:16px 20px;border-radius:8px;margin-bottom:12px'>
  <h2 style='margin:0'>🏦 CréditoBrasil — Assistente de Análise</h2>
  <p style='margin:4px 0 0 0;opacity:0.9;font-size:14px'>
    Pergunte em português sobre indicadores de crédito, inadimplência,
    juros, câmbio, inflação e PIB do Brasil.
  </p>
</div>
<p style='font-size:13px;color:#555;margin-bottom:6px'>
<b>Exemplos de perguntas:</b><br>
• Qual a inadimplência PF em São Paulo?<br>
• Quais estados têm maior saldo de crédito total?<br>
• Como o IPCA evoluiu nos últimos 12 meses?<br>
• Qual a Selic atual? | Qual o dólar hoje? | Taxa de desemprego?<br>
• Compare inadimplência PJ entre Nordeste e Sudeste
</p>
"""))

display(widgets.HBox([input_box, btn_enviar, btn_sql, btn_limpar]))
display(output)

Output()

2026/04/30 13:28:56 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['sql_query']. Expected fields: ['schema', 'question', 'error'].


---
## 🔧 Célula 10 — Otimização com DSPy BootstrapFewShot (opcional)

> Esta célula usa o otimizador do DSPy para **selecionar automaticamente os melhores exemplos** few-shot, aumentando a precisão do SQL gerado. Pode demorar alguns minutos dependendo do modelo.

In [ ]:
# ─── Métrica de avaliação: SQL executa sem erro? ──────────────────────────────
def metrica_sql_valido(exemplo, pred, trace=None):
    """Retorna 1 se o SQL gerado executa sem erro, 0 caso contrário."""
    sql = pred.sql_query if hasattr(pred, "sql_query") else ""
    _, _, erro = executar_sql(sql)
    return int(erro is None)


# ─── Dataset de avaliação ─────────────────────────────────────────────────────
DATASET_EVAL = [
    dspy.Example(
        schema=SCHEMA,
        question="Qual a inadimplência PF em SP no último mês?",
        sql_query="SELECT fst.data_referencia, fst.valor FROM fact_serie_temporal fst JOIN dim_serie ds ON ds.id_serie = fst.id_serie WHERE fst.sigla_uf = 'SP' AND ds.nome_indicador = 'Inadimplência PF - SP' ORDER BY fst.data_referencia DESC LIMIT 1"
    ).with_inputs("schema", "question"),
    dspy.Example(
        schema=SCHEMA,
        question="Qual a Selic meta atual?",
        sql_query="SELECT fst.data_referencia, fst.valor FROM fact_serie_temporal fst JOIN dim_serie ds ON ds.id_serie = fst.id_serie WHERE ds.nome_indicador = 'Selic - Meta' ORDER BY fst.data_referencia DESC LIMIT 1"
    ).with_inputs("schema", "question"),
    dspy.Example(
        schema=SCHEMA,
        question="Saldo total de crédito em RJ?",
        sql_query="SELECT fst.data_referencia, fst.valor FROM fact_serie_temporal fst JOIN dim_serie ds ON ds.id_serie = fst.id_serie WHERE fst.sigla_uf = 'RJ' AND ds.nome_indicador = 'Saldo Total - RJ' ORDER BY fst.data_referencia DESC LIMIT 1"
    ).with_inputs("schema", "question"),
    dspy.Example(
        schema=SCHEMA,
        question="IPCA acumulado em 12 meses?",
        sql_query="SELECT fst.data_referencia, fst.valor FROM fact_serie_temporal fst JOIN dim_serie ds ON ds.id_serie = fst.id_serie WHERE ds.nome_indicador = 'IPCA - Acumulado 12 meses' ORDER BY fst.data_referencia DESC LIMIT 1"
    ).with_inputs("schema", "question"),
    dspy.Example(
        schema=SCHEMA,
        question="Top 3 estados com maior inadimplência total?",
        sql_query="SELECT fst.sigla_uf, fst.valor FROM fact_serie_temporal fst JOIN dim_serie ds ON ds.id_serie = fst.id_serie WHERE ds.nome_indicador LIKE 'Inadimplência Total - %' AND fst.data_referencia = (SELECT MAX(data_referencia) FROM fact_serie_temporal fst2 JOIN dim_serie ds2 ON ds2.id_serie = fst2.id_serie WHERE ds2.nome_indicador LIKE 'Inadimplência Total - %') ORDER BY fst.valor DESC LIMIT 3"
    ).with_inputs("schema", "question"),
]


# ─── Otimizador ───────────────────────────────────────────────────────────────
from dspy.teleprompt import BootstrapFewShot

# Cria uma versão simplificada só do gerador para otimizar
class GeradorSimples(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(GerarSQL)
    def forward(self, schema, question):
        return self.prog(schema=schema, question=question)

def metrica_para_gerador(exemplo, pred, trace=None):
    sql = pred.sql_query if hasattr(pred, "sql_query") else ""
    for tag in ["```sql","```sqlite","```"]:
        sql = sql.replace(tag, "")
    sql = sql.strip().rstrip(";")
    _, _, erro = executar_sql(sql)
    return int(erro is None)

print("🔄 Iniciando otimização BootstrapFewShot...")
teleprompter = BootstrapFewShot(
    metric=metrica_para_gerador,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
)

gerador_base = GeradorSimples()
gerador_otimizado = teleprompter.compile(
    gerador_base,
    trainset=EXEMPLOS_SQL[:10],
)

# Atualiza o agente com o gerador otimizado
agente.gerador = gerador_otimizado.prog

print("✅ Otimização concluída! O agente agora usa o gerador otimizado.")
print("   Execute novamente a Célula 9 para usar o chat com o modelo otimizado.")

🔄 Iniciando otimização BootstrapFewShot...


 60%|██████    | 6/10 [00:48<00:32,  8.14s/it]

Bootstrapped 4 full traces after 6 examples for up to 1 rounds, amounting to 6 attempts.
✅ Otimização concluída! O agente agora usa o gerador otimizado.
   Execute novamente a Célula 9 para usar o chat com o modelo otimizado.


---
## 💾 Célula 11 — Salvar e carregar o agente otimizado

In [ ]:
# ─── Salvar ───────────────────────────────────────────────────────────────────
SAVE_PATH = "agente_creditobrasil.json"

agente.save(SAVE_PATH)
print(f"✅ Agente salvo em '{SAVE_PATH}'")

# Faz o download para seu computador
from google.colab import files
files.download(SAVE_PATH)

# ─── Para carregar depois ─────────────────────────────────────────────────────
# agente_recuperado = CreditoBrasilQA(exemplos=EXEMPLOS_SQL)
# agente_recuperado.load(SAVE_PATH)
# print("✅ Agente carregado!")

✅ Agente salvo em 'agente_creditobrasil.json'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 📖 Apêndice — Referência do banco de dados

### Tabelas com dados

| Tabela | Linhas | Descrição |
|--------|--------|-----------|
| `dim_serie` | 191 | Catálogo de indicadores |
| `dim_uf` | 27 | Estados brasileiros (26 UFs + DF) |
| `fact_serie_temporal` | 35.030 | Série histórica 2016–2026 |

### Categorias de indicadores

| Categoria | Subcategorias | Cobertura |
|-----------|---------------|----------|
| `credito` | inadimplência, saldos, concessões, endividamento, taxas_juros | Nacional + 27 UFs |
| `taxas_de_juros` | selic, poupança, referencias | Nacional |
| `inflacao_precos` | ipca_geral, igp | Nacional |
| `cambio` | dolar | Nacional |
| `atividade_economica` | pib | Nacional |
| `mercado_financeiro` | bolsa | Nacional |
| `emprego_e_renda` | mercado_trabalho_pnad | Nacional |

### Dicas de consulta

```sql
-- Ver todos os indicadores disponíveis
SELECT nome_indicador, categoria, periodicidade, unidade_medida, abrangencia
FROM dim_serie ORDER BY categoria, nome_indicador;

-- Ver indicadores por categoria
SELECT DISTINCT nome_indicador FROM dim_serie
WHERE categoria = 'credito' AND abrangencia = 'Brasil';

-- Período disponível
SELECT MIN(data_referencia), MAX(data_referencia) FROM fact_serie_temporal;
```